# PROJECT KIRA — PHASE 2 MEGA NOTEBOOK
**Authoritative Execution Orchestrator**

In [ ]:
# Install dependencies (PyG and dependencies for heterogeneous graphs)
!pip install torch_geometric networkx > /dev/null 2>&1


In [ ]:
import sys
import time
import logging
from pathlib import Path
import json

sys.path.append('/kaggle/input/project-kira/src')

from mcdl.research.phase2.state import CheckpointManager
from mcdl.research.phase2 import experiments as exp

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Constants
BASELINE_RUN = "run_tiny_s20260827_193f7897_40997ab"
BASELINE_COMMIT = "40997ab"
CURRENT_COMMIT = "1fc3e0a" # Will be parsed dynamically in full run

manager = CheckpointManager(run_id=f"phase2_{int(time.time())}", git_commit=CURRENT_COMMIT, baseline_run_id=BASELINE_RUN, baseline_git_commit=BASELINE_COMMIT)
logger.info("Phase 2 Orchestrator Initialized.")


In [ ]:
# CPU Checkpoint 1: Base Integrity
exp.run_s00(manager)
exp.run_s01(manager)
exp.run_a01(manager)
exp.run_a02(manager)
logger.info("CPU CHECKPOINT REACHED.")


In [ ]:
# GPU Execution block
exp.run_g01(manager)
exp.run_g02(manager)
exp.run_g04(manager)
exp.run_g05(manager)


In [ ]:
# Conditional Graph Executions
# Only run if G-01 demonstrated uplift
state = manager.get_state("G01")
if state == "COMPLETED":
    exp.run_g03(manager)
else:
    logger.info("G-01 did not complete successfully; skipping G-03.")


In [ ]:
# Time-gated Executions
time_elapsed = time.monotonic() - manager.global_start
time_remaining = (5.5 * 3600) - time_elapsed

if time_remaining > 45 * 60:
    exp.run_r01(manager)
    exp.run_llm01(manager)
else:
    logger.warning(f"SKIPPED_TIME_BUDGET: Only {time_remaining/60:.1f} mins remain before soft deadline.")


In [ ]:
# Final Synthesis
exp.run_final(manager)
logger.info("PHASE 2 NOTEBOOK EXECUTION FINISHED.")
